# pi 的对比对：采样、判分、组对（以及之后的生成把关）——Colab A100

融合 notebook（e16）的搜索和分量筛选不再用“关思考、数据集参考答案”的 NLL，改用 **pi 自己的推理里，对的比错的更受偏爱多少**。这个 notebook 准备这份数据：

1. **选题**：答案能自动判对错的公开题，按固定 commit 下载。数学约一半（MATH 全集里数论和代数的 3–5 级、整数答案），其余是 CRUXEval（读代码推返回值）、BBH（逻辑）、C-Eval（中文理科选择题）、MMLU-Pro（知识选择题）。默认对比用 600 题，另留 200 题（完全不重叠）只用来测正确率。
2. **采样**：pi 开思考，用它自己的 `generation_config` 采样参数，单条最长 8192 token。对比用的题每题 4 次，留出题每题 2 次。vLLM 跑在一张 A100 上，pi 以 FP8 加载（BF16 的 54GB 放不进 40GB）：这只影响“由谁来写这些推理”，融合 notebook 打分时用全精度。
3. **判分、组对**：只看 `</think>` 之后最后一个 `<answer>…</answer>`。同一道题里既有做对、又有做错（写完了、答案能解析）的，取长度最接近的一对（长度比不超过 2）。被截断或答案解析不出的不进对比对：它们不是“推理对或错”，而是没写完或格式不对。
4. **输出**：`pairs.jsonl`（每对两行，存 prompt 和回答的 token id，融合 notebook 逐 token 打分时和这里生成的完全一致）、`report.json`、pi 在留出题上的正确率（之后生成把关的基准）。

**生成把关（第 7 节，`STAGE = "veto"`）**：融合 notebook 选好模型后，会把候选（分量筛选的结果和全局搜索的结果）写到 Drive 并留一份 `veto/request.json`。到时把本 notebook 的 `STAGE` 改成 `"veto"` 再运行：在同一批留出题上开思考生成、判分，和 pi 比正确率；比 pi 低超过 5 个百分点的不通过。结果写回 `veto/result.json`，融合 notebook 读它决定上传哪一个。

**运行时**：Colab **A100**（40GB 即可）。采样约 4–6 小时（A100 每小时约 6.7 个计算单元），每 32 道题存一次结果到 Drive，断开后从头运行全部单元格会接着做。本地盘要放 pi 的权重约 56GB。

**注意**：真实的 27B、vLLM、这台机器上的 FP8 都没有在这里运行过；选题、判分、组对和把关规则在 CPU 上用假数据测过（`tests/test_contrast_pairs.py`）。第一次运行如果某一步报错，把输出发回来。


## 1. 配置

一般只需确认 `STAGE`。

| 项 | 说明 |
|---|---|
| `STAGE` | `"pairs"`：选题、采样、判分、组对（现在做这个）；`"veto"`：评融合 notebook 交来的候选（之后） |
| `DRIVE_DIR` | 与融合 notebook 相同的记录目录；本 notebook 的结果在其下 `contrast/` |
| `SUBJECT_REPO` / `SUBJECT_REVISION` | pi。版本留空：用融合 notebook 锁定的 commit（`subject_pin.json`） |
| `COUNTS` | 每类的题数：`pool` 用来组对，`heldout` 只测正确率 |
| `SAMPLES_POOL` / `SAMPLES_HELDOUT` | 每题采样次数 |
| `MAX_RESPONSE_TOKENS` | 单条回答（思考 + 答案）最长多少 token，超出算没写完 |
| `MAX_PAIR_RATIO` | 一对里两条回答的长度比上限 |
| `VLLM` | vLLM 的设置：FP8 加载、显存占用比例、最多同时多少条 |
| `CHUNK` | 每批多少道题，每批结束存一次 |
| `VETO_MAX_DROP` | 生成把关：候选正确率比 pi 低多少就不通过（0.05 = 5 个百分点） |


In [ ]:
import os, sys, json, shutil, subprocess, time, gc
from pathlib import Path

STAGE = "pairs"              # "pairs"（现在）或 "veto"（融合 notebook 交来候选之后）
DRIVE_DIR = "/content/drive/MyDrive/qwen38_efficientthink"
CONTRAST_DIR = DRIVE_DIR + "/contrast"
SUBJECT_REPO = "bytkim/Qwen3.8-27B-pi"
SUBJECT_REVISION = ""        # 留空：读融合 notebook 锁定的 DRIVE_DIR/subject_pin.json
LOCAL_MODEL = "/content/contrast_model"     # 本地盘：pi（或把关时的候选）的权重
COUNTS = {"math": {"pool": 300, "heldout": 100}, "code_reasoning": {"pool": 75, "heldout": 25}, "logic": {"pool": 75, "heldout": 25}, "chinese_mc": {"pool": 75, "heldout": 25}, "knowledge_mc": {"pool": 75, "heldout": 25}}
SAMPLES_POOL = 4
SAMPLES_HELDOUT = 2
MAX_RESPONSE_TOKENS = 8192
MAX_PAIR_RATIO = 2.0
THINKING = True
SEED = 20261104
VLLM = {"quantization": "fp8", "gpu_memory_utilization": 0.90, "max_num_seqs": 64}
CHUNK = 32
VETO_MAX_DROP = 0.05
INSTALL = True


## 2. 环境

挂载 Drive，检查 GPU（要 A100 一类、显存 ≥ 40GB 的卡），安装 vLLM。vLLM 会带它自己需要的 torch 版本：这一格在导入 torch 之前安装，所以不用重启运行时。


In [ ]:
from google.colab import drive as _drive
if not Path("/content/drive/MyDrive").exists():
    _drive.mount("/content/drive")
Path(CONTRAST_DIR).mkdir(parents=True, exist_ok=True)

_smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
if _smi.returncode != 0 or not _smi.stdout.strip():
    raise RuntimeError("没有 GPU：请在“运行时 → 更改运行时类型”里选 A100，然后从头运行。")
GPU_NAME, _mem = [x.strip() for x in _smi.stdout.strip().splitlines()[0].split(",")]
GPU_GIB = float(_mem) / 1024
print(f"GPU：{GPU_NAME}，显存 {GPU_GIB:.1f} GiB；本地盘剩余 {shutil.disk_usage('/content').free / 1024**3:.0f} GiB")
if GPU_GIB < 39:
    raise RuntimeError(f"显存只有 {GPU_GIB:.0f} GiB：27B 的 FP8 权重约 27GB，再加推理缓存，至少要 40GB 的 A100。请换 A100 运行时。")

def _pip(*args, timeout=1800):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True, timeout=timeout)
    if r.returncode != 0:
        raise RuntimeError("pip install " + " ".join(args) + " 失败：" + (r.stderr or r.stdout)[-800:])

if INSTALL:
    _pip("-U", "vllm")
    _pip("-U", "huggingface_hub", "pandas", "pyarrow")

import importlib.metadata as _md
VERSIONS = {}
for _p in ("vllm", "torch", "transformers", "huggingface_hub"):
    try:
        VERSIONS[_p] = _md.version(_p)
    except _md.PackageNotFoundError:
        VERSIONS[_p] = None
print("依赖：", VERSIONS)
if not VERSIONS["vllm"]:
    raise RuntimeError("vLLM 没装上：把 INSTALL 设为 True 再运行这一格")

TOK = os.environ.get("HF_TOKEN", "")
if not TOK:
    try:
        from google.colab import userdata
        TOK = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass
if not TOK:
    from getpass import getpass
    TOK = getpass("Hugging Face token（要能读 pi 和公开评测集）: ")
TOK = TOK.strip()
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
api = HfApi(token=TOK or None)


## 3. 模块

选题、判分、组对、把关规则和 vLLM 调用都在下面这个内嵌模块里；单元测试直接从本 notebook 取出它的源码。


In [ ]:
import importlib.util
CONTRAST_SOURCE = r'''"""Contrast pairs from the subject's own reasoning, for the fusion notebook's search and component stage (e16).

The subject S (pi) answers answer-checkable problems with thinking on, several times each. Where it is sometimes right
and sometimes wrong, one right and one wrong trace of the same problem make a pair. A candidate F is then scored by how
much more it prefers the right trace over the wrong one than S does:

    gain(F) = mean over pairs of [l_F(wrong) - l_F(right)] - [l_S(wrong) - l_S(right)]

where l is the mean per-token NLL of the whole response (thinking and answer). Both traces are S's own writing, so
style and format cancel; what is left is which line of reasoning F finds more likely. A guard keeps l_F on the right
traces within a few percent of S (the fusion notebook enforces it).

Also here: a held-out set (disjoint problems) on which S's accuracy with thinking on is measured, and later the same
for the candidates the fusion notebook builds (the veto: a candidate more than `max_drop` below S is not uploaded).

Pure parts (problem parsing and selection, answer extraction and grading, pairing, the veto rule) need only the
standard library. The vLLM parts run on a GPU and are imported lazily.
"""
import ast
import hashlib
import json
import math
from pathlib import Path
import random
import re

VERSION = 'contrast-2026-11-04.1'

# problems per category: `pool` are sampled `samples_pool` times to find pairs; `heldout` are for accuracy only
COUNTS = {'math': {'pool': 300, 'heldout': 100}, 'code_reasoning': {'pool': 75, 'heldout': 25},
          'logic': {'pool': 75, 'heldout': 25}, 'chinese_mc': {'pool': 75, 'heldout': 25},
          'knowledge_mc': {'pool': 75, 'heldout': 25}}
CEVAL_STEM = ('advanced_mathematics', 'college_chemistry', 'college_physics', 'college_programming',
              'computer_architecture', 'computer_network', 'discrete_mathematics', 'electrical_engineer',
              'high_school_biology', 'high_school_chemistry', 'high_school_mathematics', 'high_school_physics',
              'logic', 'metrology_engineer', 'middle_school_chemistry', 'middle_school_mathematics',
              'middle_school_physics', 'operating_system', 'probability_and_statistics')
BBH_TASKS = ('boolean_expressions', 'date_understanding', 'formal_fallacies', 'logical_deduction_five_objects',
             'multistep_arithmetic_two', 'navigate', 'object_counting', 'temporal_sequences',
             'tracking_shuffled_objects_five_objects', 'web_of_lies')
# Revisions as pinned by the fusion notebook's evaldata module; hendrycks_math is pinned on first use.
SOURCES = {
    'math': {'repo': 'EleutherAI/hendrycks_math', 'revision': '', 'subjects': ['number_theory', 'algebra'],
             'levels': [3, 4, 5], 'split': 'test'},
    'code_reasoning': {'repo': 'cruxeval-org/cruxeval', 'revision': 'b96af0450242eb4da433032b90998f25588a5d0f',
                       'files': ['test.jsonl']},
    'logic': {'repo': 'lukaemon/bbh', 'revision': '982bb89fd79532a8ac676a61fc42eb1aeec63f99',
              'files': [f'{t}/test-00000-of-00001.parquet' for t in BBH_TASKS]},
    'chinese_mc': {'repo': 'ceval/ceval-exam', 'revision': '617524a00b307ff6f9933702f724131fe12ca7ce',
                   'files': [f'{s}/val-00000-of-00001.parquet' for s in CEVAL_STEM]},
    'knowledge_mc': {'repo': 'TIGER-Lab/MMLU-Pro', 'revision': 'b189ec765aa7ed75c8acfea42df31fdae71f97be',
                     'files': ['data/test-00000-of-00001.parquet']},
}
GEN_EN = 'Think it through, then reply with only the final answer in the form <answer>...</answer> and nothing else.'
GEN_ZH = '请先思考，然后只输出最终答案，格式为 <answer>...</answer>，不要输出其他任何内容。'
GEN_KIND = {
    'number': ' The final answer must be a plain number: digits only, no units, commas or words.',
    'letter': ' The final answer is the letter of the correct option, for example <answer>C</answer>.',
    'literal': ' The final answer is the return value written as a Python literal, exactly as repr() would print it.',
    'word': ' The final answer is exactly the answer word or option label the question asks for.',
    'letter_zh': '最终答案是正确选项的字母，例如 <answer>B</answer>。',
}
LETTERS = 'ABCDEFGHIJ'
DEFAULT_SAMPLING = {'temperature': 0.6, 'top_p': 0.95, 'top_k': 20}


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(',', ':')).encode()).hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=1) + '\n', encoding='utf-8')
    tmp.replace(path)


def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in rows), encoding='utf-8')
    tmp.replace(path)


def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]


# ---------------------------------------------------------------- problems
def last_boxed(text):
    start = text.rfind('\\boxed{')
    if start < 0:
        return None
    i, depth = start + len('\\boxed{'), 1
    for j in range(i, len(text)):
        depth += {'{': 1, '}': -1}.get(text[j], 0)
        if depth == 0:
            return text[i:j]
    return None


def _thousands(s):
    """'1,234' -> '1234'; any other comma (a list of answers such as '1, 2') is kept."""
    return s.replace(',', '') if re.fullmatch(r'-?\d{1,3}(,\d{3})+(\.\d+)?', s) else s


def _key(text):
    return hashlib.sha256(' '.join(str(text).split()).encode()).hexdigest()[:16]


def math_item(row, subject):
    """A MATH problem of level 3-5 whose boxed answer is an integer."""
    match = re.search(r'(\d+)', str(row.get('level', '')))
    level = int(match[1]) if match else None
    boxed = last_boxed(str(row.get('solution', '')))
    if level is None or boxed is None:
        return None
    answer = _thousands(re.sub(r'[\s{}$]|\\[!,;]', '', boxed))
    if not re.fullmatch(r'-?\d+', answer):
        return None
    question = str(row['problem']).strip()
    return {'key': _key(question), 'group': f'{subject}/L{level}', 'level': level, 'question': question, 'lang': 'en',
            'kind': 'number', 'expected': int(answer)}


def mmlu_pro_item(row):
    options = [str(o) for o in list(row['options']) if str(o).strip() and str(o).strip() != 'N/A']
    idx = int(row['answer_index'])
    if not 4 <= len(options) <= 10 or not 0 <= idx < len(options) or str(row['answer']) != LETTERS[idx]:
        return None
    if len(options) != len(list(row['options'])):
        return None
    question = str(row['question']).strip() + '\n' + '\n'.join(f'{LETTERS[k]}. {o}' for k, o in enumerate(options))
    return {'key': str(row['question_id']), 'group': str(row.get('category', 'mmlu')), 'question': question,
            'lang': 'en', 'kind': 'letter', 'expected': LETTERS[idx], 'options': len(options)}


def ceval_item(row, subject):
    letter = str(row['answer']).strip()
    options = [str(row.get(k, '')).strip() for k in 'ABCD']
    if letter not in 'ABCD' or len(letter) != 1 or not all(options):
        return None
    question = str(row['question']).strip() + '\n' + '\n'.join(f'{k}. {o}' for k, o in zip('ABCD', options))
    return {'key': f'{subject}/{row["id"]}', 'group': subject, 'question': question, 'lang': 'zh', 'kind': 'letter_zh',
            'expected': letter, 'options': 4}


def cruxeval_item(row):
    output, code, args = str(row['output']).strip(), str(row['code']).rstrip(), str(row['input']).strip()
    try:
        ast.literal_eval(output)
        ast.parse(code)
    except (ValueError, SyntaxError, MemoryError, RecursionError):
        return None
    question = f'```python\n{code}\n```\nWhat does `f({args})` return?'
    return {'key': str(row['id']), 'group': 'cruxeval', 'question': question, 'lang': 'en', 'kind': 'literal',
            'expected': output}


def bbh_item(row, task, index):
    text, target = str(row['input']).strip(), str(row['target']).strip()
    if re.fullmatch(r'\([A-R]\)', target):
        if target not in text:
            return None
        kind, expected = 'letter', target[1]
    elif re.fullmatch(r'-?\d+', target):
        kind, expected = 'number', int(target)
    elif re.fullmatch(r'[A-Za-z]+', target):
        kind, expected = 'word', target
    else:
        return None
    return {'key': f'{task}/{index}', 'group': task, 'question': text, 'lang': 'en', 'kind': kind, 'expected': expected}


def math_files(files, subjects, split):
    """The parquet / jsonl files of these MATH subjects and split, from a repository file list."""
    out = []
    for subject in subjects:
        hits = sorted(f for f in files if f.startswith(subject + '/') and split in f.rsplit('/', 1)[-1]
                      and f.endswith(('.parquet', '.jsonl')))
        if not hits:
            raise ValueError(f'MATH：仓库里找不到 {subject} 的 {split} 文件')
        out += [(subject, f) for f in hits]
    return out


def candidates(category, fetch, reader, files=None):
    """All usable items of one category. fetch(repo, revision, filename) -> local path; reader(path) -> rows;
    files: the repository's file list (only needed for MATH)."""
    src = SOURCES[category]
    items = []
    if category == 'math':
        for subject, name in math_files(files or [], src['subjects'], src['split']):
            for row in reader(fetch(src['repo'], src['revision'], name)):
                item = math_item(row, subject)
                if item is not None and item['level'] in src['levels']:
                    items.append({**item, 'source': f'{src["repo"]}@{src["revision"][:12]}:{name}#{item["key"]}'})
        return items
    for name in src['files']:
        group = name.split('/')[0]
        for index, row in enumerate(reader(fetch(src['repo'], src['revision'], name))):
            if category == 'knowledge_mc':
                item = mmlu_pro_item(row)
            elif category == 'chinese_mc':
                item = ceval_item(row, group)
            elif category == 'code_reasoning':
                item = cruxeval_item(row)
            else:
                item = bbh_item(row, group, index)
            if item is not None:
                items.append({**item, 'source': f'{src["repo"]}@{src["revision"][:12]}:{name}#{item["key"]}'})
    return items


def prompt_text(item):
    if item['lang'] == 'zh':
        return f'{item["question"]}\n\n{GEN_ZH}{GEN_KIND[item["kind"]]}'
    return f'{item["question"]}\n\n{GEN_EN}{GEN_KIND[item["kind"]]}'


def _rank(seed, category, key):
    return hashlib.sha256(f'{seed}|{category}|{key}'.encode()).hexdigest()


def select(category, items, seed, pool, heldout, max_prompt_chars=6000):
    """Deduplicate, drop over-long prompts, interleave the groups in a seeded order; the first `pool` items are the pool,
    the next `heldout` the held-out set (disjoint by construction)."""
    seen, kept = set(), []
    for item in items:
        norm = ' '.join(item['question'].split())
        if norm in seen or len(prompt_text(item)) > max_prompt_chars:
            continue
        seen.add(norm)
        kept.append(item)
    groups = {}
    for item in sorted(kept, key=lambda x: _rank(seed, category, x['key'])):
        groups.setdefault(item['group'], []).append(item)
    order = sorted(groups, key=lambda g: _rank(seed, category, 'group:' + g))
    picked, need = [], pool + heldout
    while len(picked) < need and any(groups.values()):
        for g in order:
            if groups[g] and len(picked) < need:
                picked.append(groups[g].pop(0))
    if len(picked) < need:
        raise ValueError(f'{category}：只有 {len(picked)} 道合格题目，需要 {need} 道')
    out = []
    for n, item in enumerate(picked):
        split = 'pool' if n < pool else 'heldout'
        out.append({'pid': f'{category}-{_key(item["source"])[:10]}', 'split': split, 'category': category,
                    'prompt': prompt_text(item), **{k: v for k, v in item.items() if k != 'question'}})
    if len({p['pid'] for p in out}) != len(out):
        raise ValueError(f'{category}：题目编号重复')
    return out


def check_counts(counts):
    if set(counts) != set(COUNTS):
        raise ValueError(f'COUNTS 要给出这几类：{sorted(COUNTS)}')
    for c, v in counts.items():
        if set(v) != {'pool', 'heldout'} or any(type(x) is not int or x < 0 for x in v.values()) or v['pool'] < 1:
            raise ValueError(f'COUNTS["{c}"] 写成 {{"pool": 正整数, "heldout": 非负整数}}')
    return {c: dict(counts[c]) for c in COUNTS}


# ---------------------------------------------------------------- answers
_ANSWER = re.compile(r'<answer>(.*?)</answer>', re.S)


def extract_answer(text, thinking=True):
    """The content of the last <answer>...</answer> after the end of the thinking, or None (no closing </think> when
    thinking was on, or no answer tag)."""
    if thinking:
        if '</think>' not in text:
            return None
        text = text.rsplit('</think>', 1)[1]
    found = _ANSWER.findall(text)
    return found[-1].strip() if found else None


def _number(text):
    s = text.strip()
    boxed = last_boxed(s)
    if boxed is not None:
        s = boxed
    s = re.sub(r'\\text\{([^{}]*)\}', r'\1', s)
    s = _thousands(re.sub(r'[\s$]|\\[!,;]|\\\(|\\\)|[{}]', '', s).rstrip('.'))
    if not re.fullmatch(r'-?\d+(\.\d+)?', s):
        return None
    value = float(s)
    return int(value) if value == int(value) else value


def _letter(text, allowed):
    s = text.strip()
    m = re.match(r'^\(?([A-Z])\)?(?=$|[\s.:：、,，)])', s)
    if not m or m[1] not in allowed:
        return None
    return m[1]


def _literal(text):
    try:
        return ('value', ast.literal_eval(text.strip()))
    except (ValueError, SyntaxError, MemoryError, RecursionError, TypeError):
        return ('text', ' '.join(text.split()))


def grade(problem, pred):
    """True / False, or None when there is no parseable answer."""
    if pred is None:
        return None
    kind, expected = problem['kind'], problem['expected']
    if kind == 'number':
        value = _number(pred)
        return None if value is None else value == expected
    if kind in ('letter', 'letter_zh'):
        n = problem.get('options')                      # BBH option questions do not say how many (up to R)
        allowed = 'ABCDEFGHIJKLMNOPQR'[:int(n)] if n else 'ABCDEFGHIJKLMNOPQR'
        letter = _letter(pred, allowed)
        return None if letter is None else letter == expected
    if kind == 'literal':
        got, want = _literal(pred), _literal(expected)
        if got[0] == 'value' and want[0] == 'value':
            return got[1] == want[1] and type(got[1]) is type(want[1])
        return got[1] == want[1]
    if kind == 'word':
        return pred.strip().rstrip('.').strip().lower() == str(expected).lower()
    raise ValueError(f'不认识的答案类型：{kind}')


# ---------------------------------------------------------------- sampling bookkeeping
def chunks(problems, size):
    return [problems[i:i + size] for i in range(0, len(problems), size)]


def sample_seed(seed, pid, tag=''):
    return int(hashlib.sha256(f'{seed}|{tag}|{pid}'.encode()).hexdigest()[:8], 16)


def graded_samples(problem, outputs, thinking=True):
    """[{'k', 'text', 'response_ids', 'finish_reason', 'pred', 'correct'}] for one problem's raw outputs."""
    out = []
    for k, o in enumerate(outputs):
        finished = o['finish_reason'] == 'stop'
        pred = extract_answer(o['text'], thinking) if finished else None
        out.append({'k': k, 'text': o['text'], 'response_ids': [int(t) for t in o['token_ids']],
                    'finish_reason': o['finish_reason'], 'pred': pred, 'correct': grade(problem, pred)})
    return out


def accuracy(problems, results):
    """Per category and overall: the mean over problems of the share of correct samples (unfinished or unparseable
    samples count as wrong), with per-problem values for paired comparisons."""
    per = {}
    for p in problems:
        samples = results[p['pid']]['samples']
        per[p['pid']] = (p['category'], sum(bool(s['correct']) for s in samples) / max(1, len(samples)))
    cats = sorted({c for c, _ in per.values()})
    by = {c: sum(v for cc, v in per.values() if cc == c) / sum(1 for cc, _ in per.values() if cc == c) for c in cats}
    return {'overall': sum(by.values()) / len(by) if by else 0.0, 'by_category': by,
            'per_problem': {k: v for k, (_, v) in per.items()},
            'unfinished': sum(s['finish_reason'] != 'stop' for p in problems for s in results[p['pid']]['samples']),
            'unparseable': sum(s['finish_reason'] == 'stop' and s['correct'] is None
                               for p in problems for s in results[p['pid']]['samples'])}


# ---------------------------------------------------------------- pairs
def build_pairs(problems, results, prompt_ids, max_ratio=2.0, seed=0):
    """Rows for the fusion notebook: per pool problem with at least one right and one wrong finished, parseable sample,
    the right/wrong pair whose response lengths are closest (ratio at most max_ratio; ties by a seeded hash).
    Unfinished (cut at the token limit) and unparseable samples never enter a pair: they are neither right nor wrong
    reasoning. Returns (rows, report)."""
    rows, report = [], {'problems': 0, 'all_right': 0, 'all_wrong': 0, 'no_usable': 0, 'length_mismatch': 0, 'pairs': 0}
    by_category = {}
    for p in problems:
        if p['split'] != 'pool':
            continue
        report['problems'] += 1
        samples = results[p['pid']]['samples']
        usable = [s for s in samples if s['finish_reason'] == 'stop' and s['correct'] is not None]
        right = [s for s in usable if s['correct']]
        wrong = [s for s in usable if not s['correct']]
        if not usable:
            report['no_usable'] += 1
            continue
        if not wrong:
            report['all_right'] += 1
            continue
        if not right:
            report['all_wrong'] += 1
            continue
        best = None
        for r in right:
            for w in wrong:
                lr, lw = len(r['response_ids']), len(w['response_ids'])
                if not lr or not lw or max(lr, lw) / min(lr, lw) > max_ratio:
                    continue
                tie = hashlib.sha256(f'{seed}|{p["pid"]}|{r["k"]}|{w["k"]}'.encode()).hexdigest()
                score = (abs(math.log(lr / lw)), tie)
                if best is None or score < best[0]:
                    best = (score, r, w)
        if best is None:
            report['length_mismatch'] += 1
            continue
        _, r, w = best
        report['pairs'] += 1
        by_category[p['category']] = by_category.get(p['category'], 0) + 1
        for role, s in (('correct', r), ('wrong', w)):
            rows.append({'id': f'{p["pid"]}/{role}', 'pair': p['pid'], 'role': role, 'category': p['category'],
                         'source': p['source'], 'messages': [{'role': 'user', 'content': p['prompt']}],
                         'prompt_ids': list(prompt_ids[p['pid']]), 'response_ids': list(s['response_ids']),
                         'expected': p['expected'], 'pred': s['pred'], 'sample': s['k']})
    report['by_category'] = dict(sorted(by_category.items()))
    report['tokens'] = sum(len(r['prompt_ids']) + len(r['response_ids']) for r in rows)
    report['response_tokens'] = sum(len(r['response_ids']) for r in rows)
    return rows, report


def check_pairs(rows):
    """The fusion notebook's check of a pairs file: two rows per pair, one right and one wrong, same prompt."""
    pairs = {}
    for r in rows:
        for key in ('id', 'pair', 'role', 'category', 'prompt_ids', 'response_ids'):
            if key not in r:
                raise ValueError(f'对比对文件缺少字段 {key}')
        if r['role'] not in ('correct', 'wrong') or not r['response_ids'] or not r['prompt_ids']:
            raise ValueError(f'{r["id"]}：role 只能是 correct / wrong，且 prompt_ids、response_ids 非空')
        pairs.setdefault(r['pair'], {})
        if r['role'] in pairs[r['pair']]:
            raise ValueError(f'{r["pair"]}：同一个 role 出现两次')
        pairs[r['pair']][r['role']] = r
    for pid, two in pairs.items():
        if set(two) != {'correct', 'wrong'}:
            raise ValueError(f'{pid}：对比对不完整')
        if two['correct']['prompt_ids'] != two['wrong']['prompt_ids'] or two['correct']['category'] != two['wrong']['category']:
            raise ValueError(f'{pid}：两条回答的题目不同')
    if len({r['id'] for r in rows}) != len(rows):
        raise ValueError('对比对文件 id 重复')
    return pairs


# ---------------------------------------------------------------- veto
def paired_difference(per_a, per_b, repeats=2000, seed=0):
    """Mean of b - a over the problems both have, with a bootstrap 95% interval (problems resampled)."""
    keys = sorted(set(per_a) & set(per_b))
    if not keys:
        raise ValueError('两边没有共同的题')
    diffs = [per_b[k] - per_a[k] for k in keys]
    rng = random.Random(seed)
    boots = sorted(sum(rng.choice(diffs) for _ in diffs) / len(diffs) for _ in range(int(repeats)))
    return {'diff': sum(diffs) / len(diffs), 'ci95': [boots[int(0.025 * repeats)], boots[int(0.975 * repeats) - 1]],
            'problems': len(keys)}


def veto(subject, candidates, max_drop=0.05):
    """The fusion notebook's rule: candidates in order of preference (e.g. the component-stage choice, then the global
    search's F); the first whose overall accuracy is at most max_drop below S's is approved. Returns the decision."""
    rows = []
    for name, acc in candidates:
        cmp = paired_difference(subject['per_problem'], acc['per_problem'])
        ok = acc['overall'] >= subject['overall'] - float(max_drop)
        rows.append({'name': name, 'overall': acc['overall'], 'by_category': acc['by_category'], 'vs_subject': cmp, 'ok': ok})
    chosen = next((r['name'] for r in rows if r['ok']), None)
    return {'subject': {'overall': subject['overall'], 'by_category': subject['by_category']}, 'candidates': rows,
            'approved': chosen, 'max_drop': float(max_drop)}


# ---------------------------------------------------------------- GPU side (vLLM)
def sampling_params_from(generation_config):
    """temperature / top_p / top_k (and min_p, presence / repetition penalty when given) from the subject's own
    generation_config.json, with Qwen's thinking-mode defaults for what is missing."""
    out = dict(DEFAULT_SAMPLING)
    for key in ('temperature', 'top_p', 'top_k', 'min_p', 'presence_penalty', 'repetition_penalty'):
        if key in (generation_config or {}) and generation_config[key] is not None:
            out[key] = generation_config[key]
    return out


def chat_prompt_ids(tokenizer, prompt, thinking=True):
    text = tokenizer.apply_chat_template([{'role': 'user', 'content': prompt}], tokenize=False, add_generation_prompt=True,
                                         enable_thinking=thinking)
    return [int(t) for t in tokenizer(text, add_special_tokens=False)['input_ids']]


def make_llm(model_dir, max_model_len, gpu_memory_utilization=0.9, quantization='fp8', max_num_seqs=64, seed=0):
    """vLLM on one GPU. FP8 weights (dynamic, weight-only on Ampere): a 27B model in 40 GB."""
    from vllm import LLM
    base = dict(model=str(model_dir), tokenizer=str(model_dir), dtype='bfloat16', quantization=quantization,
                max_model_len=int(max_model_len), gpu_memory_utilization=float(gpu_memory_utilization),
                max_num_seqs=int(max_num_seqs), seed=int(seed), trust_remote_code=False)
    # Optional extras, dropped one by one if this vLLM / this architecture does not take them: no vision inputs
    # (saves memory on a multimodal checkpoint), prefix caching (the samples of one prompt share it).
    variants = [{'limit_mm_per_prompt': {'image': 0, 'video': 0}, 'enable_prefix_caching': True},
                {'enable_prefix_caching': True}, {}]
    error = None
    for extra in variants:
        try:
            return LLM(**base, **extra)
        except Exception as exc:              # noqa: BLE001 - shown, then the next variant
            error = exc
            print(f'vLLM 不接受 {sorted(extra) or "默认设置"}（{type(exc).__name__}: {str(exc)[:200]}）', flush=True)
    raise error


def generate(llm, prompt_ids, n, params, max_tokens, seeds):
    """Raw outputs per prompt: [[{'text', 'token_ids', 'finish_reason'}] * n]."""
    from vllm import SamplingParams
    sps = [SamplingParams(n=int(n), max_tokens=int(max_tokens), seed=int(s), skip_special_tokens=False, **params)
           for s in seeds]
    outs = llm.generate([{'prompt_token_ids': list(ids)} for ids in prompt_ids], sps, use_tqdm=True)
    return [[{'text': o.text, 'token_ids': list(o.token_ids), 'finish_reason': o.finish_reason} for o in out.outputs]
            for out in outs]
'''
_mod_dir = Path('/content/contrast_modules'); _mod_dir.mkdir(parents=True, exist_ok=True)
(_mod_dir / 'contrast_pairs.py').write_text(CONTRAST_SOURCE, encoding='utf-8')
_spec = importlib.util.spec_from_file_location('contrast_pairs', _mod_dir / 'contrast_pairs.py')
contrast_pairs = importlib.util.module_from_spec(_spec); sys.modules['contrast_pairs'] = contrast_pairs
_spec.loader.exec_module(contrast_pairs)
cp = contrast_pairs
print('contrast_pairs', cp.VERSION)


## 4. 锁定版本、选题

pi 的 commit 用融合 notebook 锁定的那个；评测集按固定 commit 下载（MATH 全集第一次用时锁定，记在 `contrast/dataset_pins.json`）。题目和它们的 prompt token 存到 `contrast/<指纹>/problems.jsonl`，之后的会话直接沿用。


In [ ]:
COUNTS = cp.check_counts(COUNTS)
_pin_file = Path(DRIVE_DIR) / "subject_pin.json"
_pin = json.loads(_pin_file.read_text()) if _pin_file.exists() else None
if SUBJECT_REVISION:
    SUBJECT_PIN = {"repo": SUBJECT_REPO, "revision": SUBJECT_REVISION, "subfolder": ""}
elif _pin and _pin.get("repo") == SUBJECT_REPO:
    SUBJECT_PIN = _pin
else:
    SUBJECT_PIN = {"repo": SUBJECT_REPO, "revision": api.model_info(SUBJECT_REPO).sha, "subfolder": ""}
if SUBJECT_PIN.get("subfolder"):
    raise RuntimeError("pi 的权重在子目录里：这个 notebook 还不支持，请把输出发回来")
print(f"pi：{SUBJECT_PIN['repo']} @ {SUBJECT_PIN['revision'][:12]}")

# the tokenizer, chat template and generation config first (a few MB)
snapshot_download(SUBJECT_PIN["repo"], revision=SUBJECT_PIN["revision"], local_dir=LOCAL_MODEL, token=TOK or None,
                  allow_patterns=["*.json", "*.jinja", "*.txt", "tokenizer*", "*.model"])
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)
_gc_path = Path(LOCAL_MODEL) / "generation_config.json"
SAMPLING = cp.sampling_params_from(json.loads(_gc_path.read_text()) if _gc_path.exists() else {})
print("采样参数（来自 pi 的 generation_config）：", SAMPLING)

# dataset pins
_pins_path = Path(CONTRAST_DIR) / "dataset_pins.json"
DATASET_PINS = json.loads(_pins_path.read_text()) if _pins_path.exists() else {}
for _cat, _src in cp.SOURCES.items():
    if not _src["revision"]:
        if _src["repo"] not in DATASET_PINS:
            DATASET_PINS[_src["repo"]] = api.dataset_info(_src["repo"]).sha
        _src["revision"] = DATASET_PINS[_src["repo"]]
cp.atomic_json(_pins_path, DATASET_PINS)

RUN_IDENTITY = {"version": cp.VERSION, "subject": SUBJECT_PIN, "counts": COUNTS, "seed": SEED, "samples_pool": SAMPLES_POOL,
                "samples_heldout": SAMPLES_HELDOUT, "max_response_tokens": MAX_RESPONSE_TOKENS, "thinking": THINKING,
                "sampling": SAMPLING, "sources": cp.SOURCES, "max_pair_ratio": MAX_PAIR_RATIO}
WORK = Path(CONTRAST_DIR) / cp.digest(RUN_IDENTITY)[:16]
WORK.mkdir(parents=True, exist_ok=True)
cp.atomic_json(WORK / "identity.json", RUN_IDENTITY)

_problems_path = WORK / "problems.jsonl"
if _problems_path.exists():
    PROBLEMS = cp.read_jsonl(_problems_path)
    print("沿用已选好的题：", _problems_path)
else:
    import pandas as pd
    _data_dir = Path("/content/contrast_data")
    def _fetch(repo, revision, name):
        return hf_hub_download(repo, name, repo_type="dataset", revision=revision, local_dir=str(_data_dir / repo.replace("/", "__")),
                               token=TOK or None)
    def _reader(path):
        path = str(path)
        return cp.read_jsonl(path) if path.endswith(".jsonl") else pd.read_parquet(path).to_dict("records")
    PROBLEMS = []
    for _cat in cp.SOURCES:
        _files = None
        if _cat == "math":
            _files = api.list_repo_files(cp.SOURCES[_cat]["repo"], repo_type="dataset", revision=cp.SOURCES[_cat]["revision"])
        _items = cp.candidates(_cat, _fetch, _reader, _files)
        _picked = cp.select(_cat, _items, SEED, COUNTS[_cat]["pool"], COUNTS[_cat]["heldout"])
        print(f"{_cat}：{len(_items)} 道可用，选 {len(_picked)} 道")
        PROBLEMS += _picked
    for _p in PROBLEMS:
        _p["prompt_ids"] = cp.chat_prompt_ids(tokenizer, _p["prompt"], THINKING)
    cp.write_jsonl(_problems_path, PROBLEMS)
PROMPT_IDS = {p["pid"]: p["prompt_ids"] for p in PROBLEMS}
_lens = sorted(len(v) for v in PROMPT_IDS.values())
print(f"题目：对比用 {sum(p['split'] == 'pool' for p in PROBLEMS)} 道，留出 {sum(p['split'] == 'heldout' for p in PROBLEMS)} 道；"
      f"prompt 长度中位 {_lens[len(_lens) // 2]}、最长 {_lens[-1]} token")
print("示例 prompt：\n" + tokenizer.decode(PROMPT_IDS[PROBLEMS[0]["pid"]])[:1500])
MAX_MODEL_LEN = _lens[-1] + MAX_RESPONSE_TOKENS + 16


## 5. 加载 pi（vLLM，FP8）

下载 pi 的完整权重（约 56GB）到本地盘，用 vLLM 以 FP8 加载。`STAGE = "veto"` 时跳过这一节（把关时加载的是候选）。


In [ ]:
LLM = None
if STAGE == "pairs":
    snapshot_download(SUBJECT_PIN["repo"], revision=SUBJECT_PIN["revision"], local_dir=LOCAL_MODEL, token=TOK or None)
    LLM = cp.make_llm(LOCAL_MODEL, MAX_MODEL_LEN, VLLM["gpu_memory_utilization"], VLLM["quantization"], VLLM["max_num_seqs"], SEED)
    print("vLLM 已加载 pi；最长上下文", MAX_MODEL_LEN)


## 6. 采样、判分、组对

对比题每题 `SAMPLES_POOL` 次、留出题每题 `SAMPLES_HELDOUT` 次；每 `CHUNK` 道题存一次（`samples/`），断开后从头运行全部单元格会跳过已完成的批。每道题的随机种子固定，重跑结果相同。全部采完后判分、组对，写出 `pairs.jsonl` 和 `report.json`。


In [ ]:
def run_samples(llm, problems, split, n, out_dir, tag):
    """Sample every problem of one split in chunks; returns {pid: {'samples': [...]}}; finished chunks are reused."""
    todo = [p for p in problems if p["split"] == split]
    results = {}
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    batches = cp.chunks(todo, CHUNK)
    started = time.perf_counter(); made = 0; tokens = 0
    for i, batch in enumerate(batches):
        path = out_dir / f"{split}-{i:03d}.jsonl"
        if path.exists():
            for row in cp.read_jsonl(path):
                results[row["pid"]] = row
            continue
        outs = cp.generate(llm, [PROMPT_IDS[p["pid"]] for p in batch], n, SAMPLING, MAX_RESPONSE_TOKENS,
                           [cp.sample_seed(SEED, p["pid"], tag) for p in batch])
        rows = []
        for p, o in zip(batch, outs):
            rows.append({"pid": p["pid"], "category": p["category"], "samples": cp.graded_samples(p, o, THINKING)})
            tokens += sum(len(x["token_ids"]) for x in o)
        cp.write_jsonl(path, rows)
        for row in rows:
            results[row["pid"]] = row
        made += 1
        el = time.perf_counter() - started
        left = sum(1 for j in range(i + 1, len(batches)) if not (out_dir / f"{split}-{j:03d}.jsonl").exists())
        print(f"{split} 第 {i + 1}/{len(batches)} 批：本次已生成 {tokens / 1e6:.2f}M token，{tokens / el:.0f} token/s，"
              f"估计还需 {el / made * left / 60:.0f} 分钟", flush=True)
    return results

if STAGE == "pairs":
    RESULTS = {}
    RESULTS.update(run_samples(LLM, PROBLEMS, "pool", SAMPLES_POOL, WORK / "samples", "pool"))
    RESULTS.update(run_samples(LLM, PROBLEMS, "heldout", SAMPLES_HELDOUT, WORK / "samples", "heldout"))
    POOL = [p for p in PROBLEMS if p["split"] == "pool"]
    HELDOUT = [p for p in PROBLEMS if p["split"] == "heldout"]
    ACC_POOL = cp.accuracy(POOL, RESULTS)
    ACC_HELDOUT = cp.accuracy(HELDOUT, RESULTS)
    ROWS, PAIR_REPORT = cp.build_pairs(PROBLEMS, RESULTS, PROMPT_IDS, MAX_PAIR_RATIO, SEED)
    cp.check_pairs(ROWS)
    cp.write_jsonl(WORK / "pairs.jsonl", ROWS)
    cp.atomic_json(WORK / "heldout_subject.json", {"identity": RUN_IDENTITY, "accuracy": ACC_HELDOUT})
    REPORT = {"identity": RUN_IDENTITY, "versions": VERSIONS, "gpu": GPU_NAME, "pairs": PAIR_REPORT,
              "accuracy_pool": {k: v for k, v in ACC_POOL.items() if k != "per_problem"},
              "accuracy_heldout": {k: v for k, v in ACC_HELDOUT.items() if k != "per_problem"}}
    cp.atomic_json(WORK / "report.json", REPORT)
    print(json.dumps({k: v for k, v in REPORT.items() if k not in ("identity",)}, ensure_ascii=False, indent=1))
    print(f"\n对比对：{PAIR_REPORT['pairs']} 对，回答共 {PAIR_REPORT['response_tokens'] / 1e6:.2f}M token。")
    print("在融合 notebook（e16）第 1 节填：")
    print(f'CONTRAST_PAIRS_FILE = "{WORK / "pairs.jsonl"}"')


## 7. 生成把关（`STAGE = "veto"`）

融合 notebook 选好模型后，会把候选写到 Drive 并留下 `DRIVE_DIR/veto/request.json`。把第 1 节的 `STAGE` 改成 `"veto"`，在 A100 上从头运行：候选按偏好顺序（先分量筛选的结果，再全局搜索的结果）一个一个评：复制到本地盘、用 vLLM（FP8，和 pi 相同）在同一批留出题上、用同样的种子各生成 `SAMPLES_HELDOUT` 次，判分，和 pi 比正确率。第一个“比 pi 低不超过 5 个百分点”的通过，后面的就不评了，结果写进 `veto/result.json`；融合 notebook 读它决定上传哪个。

**一次运行只加载一个候选**：vLLM 在同一个进程里先后加载两个 27B 模型时，显存不一定能释放干净。第一个候选没通过、还要评第二个时，本节会停下并提示：用“代码执行程序 → 重启会话”后再从头运行，已评完的候选（`veto/eval-<名字>.json`）和已生成的批次（`veto/samples/`）都会沿用。


In [ ]:
if STAGE == "veto":
    _req_path = Path(DRIVE_DIR) / "veto" / "request.json"
    if not _req_path.exists():
        raise RuntimeError(f"没有 {_req_path}：融合 notebook 还没有交来候选")
    REQUEST = json.loads(_req_path.read_text())
    _subject_path = WORK / "heldout_subject.json"
    if not _subject_path.exists():
        raise RuntimeError("还没有 pi 在留出题上的结果：先用 STAGE = \"pairs\" 跑完采样")
    SUBJECT_ACC = json.loads(_subject_path.read_text())["accuracy"]
    HELDOUT = [p for p in PROBLEMS if p["split"] == "heldout"]
    EVALS = []
    DECISION = None
    _loaded = False
    for cand in REQUEST["candidates"]:              # in order of preference; the first that passes is taken
        out = _req_path.parent / f"eval-{cand['name']}.json"
        saved = json.loads(out.read_text()) if out.exists() else None
        if saved is not None and saved.get("fingerprint") == cand["fingerprint"]:
            acc = saved["accuracy"]
            print(f"{cand['name']}：沿用已评完的结果，正确率 {acc['overall']:.3f}")
        else:
            if _loaded:
                # vLLM does not reliably give the GPU back within one process: one candidate per run.
                print(f"\n{cand['name']} 还没评：请用“代码执行程序 → 重启会话”，再从头运行本 notebook（STAGE 仍是 \"veto\"）。"
                      "已评完的候选和已生成的批次都会沿用。")
                break
            local = Path("/content/veto_model")
            shutil.rmtree(local, ignore_errors=True)
            print(f"{cand['name']}：从 {cand['directory']} 复制到本地盘…", flush=True)
            shutil.copytree(cand["directory"], local)
            llm = cp.make_llm(local, MAX_MODEL_LEN, VLLM["gpu_memory_utilization"], VLLM["quantization"], VLLM["max_num_seqs"], SEED)
            _loaded = True
            results = run_samples(llm, HELDOUT, "heldout", SAMPLES_HELDOUT,
                                  _req_path.parent / "samples" / f"{cand['name']}-{cand['fingerprint'][:12]}", "heldout")
            acc = cp.accuracy(HELDOUT, results)
            cp.atomic_json(out, {"name": cand["name"], "fingerprint": cand["fingerprint"], "accuracy": acc, "versions": VERSIONS})
            print(f"{cand['name']}：正确率 {acc['overall']:.3f}（pi {SUBJECT_ACC['overall']:.3f}）", flush=True)
        EVALS.append((cand["name"], acc))
        DECISION = cp.veto(SUBJECT_ACC, EVALS, VETO_MAX_DROP)
        if DECISION["approved"]:
            break
    if DECISION is not None and (DECISION["approved"] or len(EVALS) == len(REQUEST["candidates"])):
        DECISION.update(request=REQUEST, versions=VERSIONS)
        cp.atomic_json(_req_path.parent / "result.json", DECISION)
        print(json.dumps({k: v for k, v in DECISION.items() if k != "request"}, ensure_ascii=False, indent=1))
        print("\n通过：" + (DECISION["approved"] or "没有候选通过；先不要上传，把上面的输出发回来"))
